# Hotel cancellation and overbooking baseline

Thin client: loading, features, training, scoring and policy calculations live in `hotel_cancel`. The original teaching notebook is preserved unchanged in `reference/`.


In [ ]:
from IPython.display import display
from hotel_cancel.config import load_config
from hotel_cancel.train import run_baseline

config = load_config()
result = run_baseline(config)


## Data and forward splits

In [ ]:
display(result.bookings.head(3))
display({name: int(mask.sum()) for name, mask in result.masks.items()})


## Per-hotel results and diagnostic gate

The supplied baseline is expected to pass the City gate and fail Resort calibration. This is evidence to document, not a reason to tune against the holdout.

In [ ]:
display(result.metrics)
display(result.gate)
print(f"Runtime: {result.runtime_seconds:.1f}s")
print(f"Holdout SHA-256: {result.fingerprint}")


## Booking-time feature importance

In [ ]:
import pandas as pd

importance = pd.Series(result.model.booster_.feature_importance("gain"), index=result.features.columns)
(100 * importance / importance.sum()).nlargest(12).sort_values().plot.barh(
    title="Share of feature gain (%)", figsize=(8, 4)
)


## Given overbooking policy

Calibration uses Jul–Sep 2016; policy selection uses Oct–Dec 2016. Test results are reported for the frozen policies.

In [ ]:
from hotel_cancel.policy import run_policy_baseline

policy = run_policy_baseline(result, config)
display(policy.selection)
display(policy.chosen)
display(policy.summary)
display(pd.DataFrame(policy.tonight).T)
print(f"Policy runtime: {policy.runtime_seconds:.1f}s")